# Skin Cancer Detection — Colab Training

Trains **6 models** on HAM10000 dataset.

| # | Model | Type | Approx Time |
|---|---|---|---|
| 1 | Sequential CNN | From scratch | ~10 min |
| 2 | Custom ResNet | From scratch | ~15 min |
| 3 | EfficientNetB0 | Transfer learning | ~35 min |
| 4 | ResNet50 | Transfer learning | ~30 min |
| 5 | DenseNet121 | Transfer learning | ~30 min |
| 6 | ViT | Transfer learning | ~45 min |

> **⚠️ IMPORTANT**: For transfer learning models (3–6), **restart the runtime** before each one:
> `Runtime → Restart runtime`, then re-run **Setup Cell** below, then the model cell.
> This is needed because 224×224 images use ~6 GB RAM, and leftover data from
> a previous model causes out-of-memory crashes.

Models 1 & 2 (from-scratch) can run back-to-back without restarting.

## ⚙️ Setup Cell (run this after every runtime restart)

In [ ]:
# === SETUP: Run this cell after every runtime restart ===
import os, sys

# Clone or update repo
if not os.path.exists('/content/SkinCancer'):
    !git clone https://github.com/Ashutosh-Repos/SkinCancer.git /content/SkinCancer
else:
    !cd /content/SkinCancer && git pull

%cd /content/SkinCancer

# Install deps (quiet)
!pip install flask flask-cors python-dotenv tqdm tensorflow-hub -q

# Add src to path
sys.path.insert(0, '/content/SkinCancer/src')

# GPU check
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print(f'TensorFlow {tf.__version__} | GPU: {gpus[0].name if gpus else "NONE!"}')

# Ensure directories
for d in ['models/checkpoints', 'logs', 'results']:
    os.makedirs(d, exist_ok=True)

# Check dataset
if os.path.exists('data/images'):
    n = len([f for f in os.listdir('data/images') if f.endswith('.jpg')])
    print(f'Dataset: {n} images ✅' if n >= 10000 else f'Dataset: only {n} images ⚠️')
else:
    print('Dataset: NOT FOUND — run the Download cell below')

print('Setup complete!')

## 📦 Download Dataset (only need to run once)

In [ ]:
import os
if not os.path.exists('data/images') or len(os.listdir('data/images')) < 10000:
    token = input('Paste your Kaggle API Token (KGAT_xxx): ')
    os.environ['KAGGLE_API_TOKEN'] = token.strip()
    !python scripts/download_dataset.py
else:
    print('Dataset already exists!')

if os.path.exists('data/images'):
    print(f"Images: {len([f for f in os.listdir('data/images') if f.endswith('.jpg')])}")
else:
    print('ERROR: Dataset download failed. Check your Kaggle token and try again.')

## 💾 Mount Google Drive (run once per session)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/SkinCancer'
for d in ['models/checkpoints', 'logs', 'results']:
    os.makedirs(f'{DRIVE_DIR}/{d}', exist_ok=True)
print(f'Drive mounted: {DRIVE_DIR}')

---
## 🏋️ Model Training

> Run models 1 & 2 back-to-back. For models 3–6, **restart runtime** before each.

---

In [ ]:
# Helper: Save results to Drive after each model
import shutil

def save_to_drive(model_name):
    """Copy model checkpoint & results to Google Drive."""
    DRIVE_DIR = '/content/drive/MyDrive/SkinCancer'
    # Copy checkpoint
    for ext in ['.h5', '_metadata.json']:
        src = f'models/checkpoints/{model_name}_best{ext}'
        if os.path.exists(src):
            shutil.copy(src, f'{DRIVE_DIR}/models/checkpoints/')
    # Copy logs (find latest)
    import glob
    log_dirs = sorted(glob.glob(f'logs/{model_name}_*'))
    if log_dirs:
        dst = f'{DRIVE_DIR}/logs/{os.path.basename(log_dirs[-1])}'
        shutil.copytree(log_dirs[-1], dst, dirs_exist_ok=True)
    # Copy norm_stats
    if os.path.exists('data/norm_stats.json'):
        shutil.copy('data/norm_stats.json', f'{DRIVE_DIR}/')
    print(f'  ✅ {model_name} saved to Drive')

print('Helper function loaded.')

### Model 1: Sequential CNN (~10 min)
Can run back-to-back with Model 2.

In [ ]:
!python src/train.py --model sequential --epochs 30 --batch-size 64
save_to_drive('sequential')

### Model 2: Custom ResNet (~15 min)
Can run right after Model 1.

In [ ]:
!python src/train.py --model resnet --epochs 16 --batch-size 64
save_to_drive('resnet')

### Model 3: EfficientNetB0 (~35 min)
> ⚠️ **Restart runtime first!** `Runtime → Restart runtime`, then re-run **Setup Cell** + **Mount Drive** + **Helper function cell**.

In [ ]:
!python src/train.py --model efficientnet --batch-size 16
save_to_drive('efficientnet')

### Model 4: ResNet50 (~30 min)
> ⚠️ **Restart runtime first!**

In [ ]:
!python src/train.py --model resnet50 --batch-size 16
save_to_drive('resnet50')

### Model 5: DenseNet121 (~30 min)
> ⚠️ **Restart runtime first!**

In [ ]:
!python src/train.py --model densenet --batch-size 16
save_to_drive('densenet')

### Model 6: Vision Transformer (~45 min)
> ⚠️ **Restart runtime first!**

In [ ]:
!python src/train.py --model vit --batch-size 16
save_to_drive('vit')

---
## 🧪 Class Imbalance Experiment
> ⚠️ **Restart runtime first!**

Backs up base EfficientNet, then retrains with class weights.

---

In [ ]:
import shutil, os

# Step 1: Backup base EfficientNet checkpoint
for ext in ['.h5', '_metadata.json']:
    src = f'models/checkpoints/efficientnet_best{ext}'
    dst = f'models/checkpoints/efficientnet_no_cw_best{ext}'
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f'  Backed up {src}')

# Step 2: Train with class weights
!python src/train.py --model efficientnet --batch-size 16 --class-weights
save_to_drive('efficientnet')
save_to_drive('efficientnet_no_cw')

---
## 📊 Evaluate All Models
> ⚠️ **Restart runtime first!** Evaluation also loads images.

---

In [ ]:
import glob, os

model_files = sorted(glob.glob('models/checkpoints/*_best.h5'))
print(f'Found {len(model_files)} models:')
for f in model_files:
    mb = os.path.getsize(f) / (1024*1024)
    print(f'  {os.path.basename(f):40s} ({mb:.1f} MB)')

for mf in model_files:
    name = os.path.basename(mf)
    results_dir = f"results/{name.replace('.h5', '')}"
    if os.path.exists(f'{results_dir}/evaluation_metrics.json'):
        print(f'\nSkipping {name} (already evaluated)')
        continue
    print(f"\n{'='*60}")
    print(f'Evaluating: {name}')
    print(f"{'='*60}")
    !cd /content/SkinCancer && python src/evaluate.py --model {mf}

## 📋 Model Comparison Table

In [ ]:
import json, glob, pandas as pd

results = []
for mf in sorted(glob.glob('results/*/evaluation_metrics.json')):
    name = mf.split('/')[-2]
    with open(mf) as f:
        m = json.load(f)
    results.append({
        'Model': name,
        'Accuracy': f"{m['accuracy']*100:.2f}%",
        'F1 (Macro)': f"{m['f1_macro']*100:.2f}%",
        'F1 (Weighted)': f"{m['f1_weighted']*100:.2f}%",
        'Precision': f"{m['precision_macro']*100:.2f}%",
        'Recall': f"{m['recall_macro']*100:.2f}%",
    })

if results:
    df = pd.DataFrame(results)
    display(df)
    df.to_csv('results/model_comparison.csv', index=False)
    print('\nSaved to results/model_comparison.csv')
else:
    print('No results found.')

## 📈 Training Curves & Confusion Matrices

In [ ]:
from IPython.display import Image as IPImage, display
import glob

print('TRAINING CURVES')
for p in sorted(glob.glob('logs/*/training_history.png')):
    print(f'\n--- {p.split("/")[-2]} ---')
    display(IPImage(filename=p, width=800))

print('\nCONFUSION MATRICES')
for p in sorted(glob.glob('results/*/confusion_matrix.png')):
    print(f'\n--- {p.split("/")[-2]} ---')
    display(IPImage(filename=p, width=800))

## 💾 Save Everything to Drive

In [ ]:
import shutil
DRIVE_DIR = '/content/drive/MyDrive/SkinCancer'
!cp -r models/checkpoints/ {DRIVE_DIR}/models/checkpoints/
!cp -r logs/ {DRIVE_DIR}/logs/
!cp -r results/ {DRIVE_DIR}/results/
!cp data/norm_stats.json {DRIVE_DIR}/ 2>/dev/null || true
print('All outputs saved to Google Drive!')
!find {DRIVE_DIR} -name '*.h5' -o -name '*.json' -o -name '*.png' -o -name '*.csv' | head -30

## ⬇️ Download Best Model

In [ ]:
from google.colab import files
for f in ['models/checkpoints/efficientnet_best.h5',
          'models/checkpoints/efficientnet_best_metadata.json',
          'data/norm_stats.json',
          'results/model_comparison.csv']:
    if os.path.exists(f):
        files.download(f)
        print(f'Downloaded: {f}')
print('\nUsage on MacBook:')
print('  python src/inference.py --model models/checkpoints/efficientnet_best.h5 --image <image.jpg>')